# Season features for Lacuna 2023: mean, pre-harvest, peak

**Question.** Which way of turning the monthly greenness series into season numbers ranks Lacuna fields best?

| Variant | Features (per index: NDVI, GCVI, MTCI, NDVI705, NDVI740) |
|---|---|
| V0 | mean over the whole season window (Oct 2022 – Jun 2023) |
| V1 | mean over a fixed pre-harvest window, **Nov – Apr** (crop calendar, chosen before scoring; no per-field survey dates) |
| V2 | max (= the peak value) |
| V2+ | peak features from each field's own curve, for GCVI, NDVI705, MTCI: peak value, peak month, area under the curve before the peak |
| V3 | mean + median + max together (a mode makes no sense for continuous values, so the median replaces it) |
| A1 | the current baseline: max + mean (10 features) |

**Rules.** Same fields (1,880, at least 4 clear months), same model (Ridge with clipping and scaling inside each fold), same
2 km folds. Folds 1–4 are scored; fold-0 fields only train. Every variant is compared with A1 on the same fields by a paired
cluster bootstrap (seed 20261029). A variant **wins** if its rank-inside-clusters interval vs A1 lies above 0 and its RMSE
interval is not entirely worse. Six comparisons, so one "win" can be luck: treat this as exploratory.

In [1]:
import importlib.util
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
V2 = REPO / "data" / "v2"
spec = importlib.util.spec_from_file_location("step8", REPO / "scripts" / "baselines" / "baseline_table.py")
step8 = importlib.util.module_from_spec(spec); spec.loader.exec_module(step8)
from cropyield.eval.scores import centred, in_large_clusters, paired_difference_ci, r2, rank_inside_clusters, rmse, spearman

COHORT = "zimbabwe_lacuna_2022_2023"
labels = pd.read_parquet(V2 / "processed" / "labels_v2.parquet", columns=["obs_id", "yield_kg_ha", "lat", "lon"]).set_index("obs_id")
X_a1, y, fields = step8.cohort_rows(COHORT, V2, labels, step8.COHORTS_V2[COHORT], "labels_v2")
print(len(y), "fields | folds:", fields["fold"].value_counts().sort_index().to_dict())

1880 fields | folds: {'0': 376, '1': 376, '2': 376, '3': 376, '4': 376}


In [2]:
# The monthly table: one row per field and month, index values only for clear months
INDICES = ["ndvi", "gcvi", "mtci", "ndvi705", "ndvi740"]
with duckdb.connect(str(V2 / "cropyield.duckdb"), read_only=True) as con:
    monthly = con.execute(f"""SELECT obs_id, month, {', '.join(INDICES)} FROM monthly
                              WHERE cohort = ? AND coverage_state = 'clear'""", [COHORT]).df()
monthly = monthly[monthly.obs_id.isin(y.index)]
MONTHS = sorted(monthly.month.unique())
cube = {i: monthly.pivot(index="obs_id", columns="month", values=i).reindex(index=y.index, columns=MONTHS) for i in INDICES}
print("clear months:", MONTHS)
print("share of fields clear per month:", cube["ndvi"].notna().mean().round(2).to_dict())

clear months: ['2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06']
share of fields clear per month: {'2022-12': 0.98, '2023-01': 0.82, '2023-02': 0.76, '2023-03': 0.94, '2023-04': 1.0, '2023-05': 1.0, '2023-06': 1.0}


In [3]:
PRE_HARVEST = [m for m in MONTHS if "2022-11" <= m <= "2023-04"]       # Nov - Apr, fixed for every field
STEP = {m: k for k, m in enumerate(MONTHS)}                              # month number for timing and areas


def peak_features(series: pd.Series) -> tuple[float, float, float]:
    """Peak value, peak month (0 = first month with photos) and area before the peak (gaps filled linearly)."""
    clear = series.dropna()
    x, v = np.array([STEP[m] for m in clear.index]), clear.to_numpy()
    peak = int(np.argmax(v))
    grid = np.arange(x[0], x[peak] + 1)
    curve = np.interp(grid, x, v)                                        # one value per month, gaps filled
    return v[peak], float(x[peak]), float(((curve[:-1] + curve[1:]) / 2).sum())   # trapezoid area, 0 if peak first


full_mean = pd.DataFrame({f"{i}_mean": cube[i].mean(axis=1) for i in INDICES})
pre = pd.DataFrame({f"{i}_mean_nov_apr": cube[i][PRE_HARVEST].mean(axis=1) for i in INDICES})
print("fields without a clear Nov-Apr month (use their season mean):", int(pre.isna().any(axis=1).sum()))
pre = pre.fillna(full_mean.set_axis(pre.columns, axis=1))
peaks = pd.DataFrame({f"{i}_{name}": vals for i in ["gcvi", "ndvi705", "mtci"]
                      for name, vals in zip(["peak", "peak_month", "area_before_peak"],
                                            zip(*[peak_features(row) for _, row in cube[i].iterrows()]))},
                     index=y.index)
variants = {
    "V0 mean, whole season": full_mean,
    "V1 mean, Nov-Apr": pre,
    "V2 max (peak value)": pd.DataFrame({f"{i}_max": cube[i].max(axis=1) for i in INDICES}),
    "V2+ peak, peak month, area before peak": peaks,
    "V3 mean + median + max": pd.concat([full_mean, pd.DataFrame({f"{i}_median": cube[i].median(axis=1) for i in INDICES}),
                                         pd.DataFrame({f"{i}_max": cube[i].max(axis=1) for i in INDICES})], axis=1),
    "A1 max + mean (baseline)": X_a1,
}
# Check: the max and mean built here equal the database's season features
assert np.allclose(variants["V2 max (peak value)"].to_numpy(), X_a1[[f"{i}_season_max" for i in INDICES]].to_numpy())
assert np.allclose(full_mean.to_numpy(), X_a1[[f"{i}_season_mean" for i in INDICES]].to_numpy())
peaks.describe().round(2)

fields without a clear Nov-Apr month (use their season mean): 0


,gcvi_peak,gcvi_peak_month,gcvi_area_before_peak,ndvi705_peak,ndvi705_peak_month,ndvi705_area_before_peak,mtci_peak,mtci_peak_month,mtci_area_before_peak
count,1880.00,1880.00,1880.00,1880.00,1880.00,1880.00,1880.00,1880.00,1880.00
mean,3.64,2.01,5.02,0.46,1.90,0.64,7.20,2.40,6.60
std,1.24,0.98,2.63,0.11,0.91,0.32,23.20,2.14,45.07
min,1.21,0.00,0.00,0.14,0.00,0.00,1.18,0.00,-1445.96
25%,2.72,1.00,3.24,0.38,1.00,0.42,3.29,1.00,2.33
50%,3.45,2.00,4.88,0.46,2.00,0.64,4.12,2.00,5.43
75%,4.37,3.00,6.63,0.54,3.00,0.85,5.52,4.00,11.87
max,9.81,5.00,17.79,0.73,5.00,1.72,627.26,6.00,339.32


In [4]:
tested = (fields["fold"] != "0").to_numpy()
yt, ft = y.to_numpy()[tested], fields[tested]
groups = ft["group_id"].to_numpy()
cy = (ft["group_id"] + "|" + ft["season_year"].astype(str)).to_numpy()
keep = in_large_clusters(cy, step8.MIN_CLUSTER_FIELDS)

predictions = {}
for name, X in variants.items():
    out = step8.out_of_fold(X.loc[y.index], y, fields, skip=("0",))
    predictions[name] = out["A1"][tested]
predictions["R1 coordinates only"] = out["R1"][tested]

base = predictions["A1 max + mean (baseline)"]
rows = []
for name, p in predictions.items():
    row = {"variant": name, "features": variants[name].shape[1] if name in variants else 2,
           "RMSE t/ha": rmse(yt, p), "R2": r2(yt, p), "rank inside clusters": rank_inside_clusters(yt, p, cy, step8.MIN_CLUSTER_FIELDS)}
    if name != "A1 max + mean (baseline)":
        y_in, p_in, b_in = (centred(v[keep], cy[keep]) for v in (yt, p, base))
        lo, hi = paired_difference_ci(y_in, p_in, b_in, groups[keep], spearman, seed=step8.SEED)
        rlo, rhi = paired_difference_ci(yt, p, base, groups, rmse, seed=step8.SEED)
        row |= {"rank vs A1": f"{spearman(y_in, p_in) - spearman(y_in, b_in):+.3f} ({lo:+.3f} to {hi:+.3f})",
                "RMSE vs A1": f"{rmse(yt, p) - rmse(yt, base):+.3f} ({rlo:+.3f} to {rhi:+.3f})",
                "wins": "yes" if lo > 0 and not rlo > 0 else "no"}
    rows.append(row)
results = pd.DataFrame(rows).set_index("variant").round(3)
print(f"Lacuna 2023, folds 1-4: {tested.sum()} fields, {len(set(groups))} clusters; rank uses {keep.sum()} fields in clusters of >= 3")
results

Lacuna 2023, folds 1-4: 1504 fields, 232 clusters; rank uses 1342 fields in clusters of >= 3


,features,RMSE t/ha,R2,rank inside clusters,rank vs A1,RMSE vs A1,wins
variant,,,,,,,
"V0 mean, whole season",5,1.857,0.124,0.262,-0.037 (-0.072 to -0.003),+0.072 (+0.029 to +0.110),no
"V1 mean, Nov-Apr",5,1.826,0.153,0.296,-0.003 (-0.037 to +0.031),+0.040 (+0.000 to +0.075),no
V2 max (peak value),5,1.796,0.181,0.307,+0.008 (-0.011 to +0.026),+0.011 (-0.011 to +0.033),no
"V2+ peak, peak month, area before peak",9,1.765,0.209,0.319,+0.020 (-0.018 to +0.059),-0.020 (-0.054 to +0.014),no
V3 mean + median + max,15,1.784,0.191,0.300,+0.001 (-0.007 to +0.008),-0.001 (-0.009 to +0.007),no
A1 max + mean (baseline),10,1.785,0.191,0.299,NaN,NaN,NaN
R1 coordinates only,2,1.854,0.127,-0.054,-0.353 (-0.433 to -0.260),+0.069 (-0.082 to +0.217),no


**How to read it.** R² and RMSE are about t/ha; *rank inside clusters* asks "is this field better than the others in its
village?" (0.30 means the model picks the better of two fields in a village about 60 % of the time). "rank vs A1" and
"RMSE vs A1" are paired differences with 95 % intervals; an interval that contains 0 means "no clear difference".

**Caveats.** Exploratory: six comparisons on the same folds. Fold 0 is untouched, so a winner can be confirmed there once at
the end (train and tune on folds 1–4, predict fold 0 once). October and November 2022 have no photos at all (catalogue gap),
so "Nov–Apr" really starts with the first December photo.